intro_to_Language_modeling part4 Continuation..
------

[video link](https://youtu.be/q8SA3rM6ckI)

<ins>Goal</ins>

Understand backprop working - manual backprop instead of using PyTorch autograd's loss.backward():

Initial Setup
-----

In [35]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [36]:
# # optional - using exact version of pythorch from video
# %pip install --force-reinstall -v "torch==1.13.1" --quiet
# print(torch. __version__)

# # optional - debugging config
# %pip install ipdb --quiet
# import os
# os.environ['PYTHONBREAKPOINT'] = 'ipdb.set_trace'

In [37]:
# download the training data
!curl https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt -o names.txt -s

In [38]:
words = open('names.txt', 'r').read().splitlines()

In [39]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [40]:
# build the dataset for training,dev and test split
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):
  X, Y = [], []

  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80% training split
Xdev, Ydev = build_dataset(words[n1:n2])   # 10% dev split
Xte,  Yte  = build_dataset(words[n2:])     # 10% test split

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [41]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t): # here 'dt' is the manual derivative and 't' is the derivative calculated by pytorch
  ex = torch.all(dt == t.grad).item() # check if manual and pytorch derivative values are exactly equal
  app = torch.allclose(dt, t.grad) # check if manual and pytorch derivative values are close enough
  maxdiff = (dt - t.grad).abs().max().item() # if there is diff in manual and pytorch derivative, what is the diff
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

Init Network
----

In [42]:
#init 2 layer MLP network

n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP
g = torch.Generator().manual_seed(2147483647) # for reproducibility

# embedding tables for chars
C  = torch.randn((vocab_size, n_embd),            generator=g)

# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1 # using b1 just for fun, ideally it's useless when we use BatchNorm

# BatchNorm
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1 # setting to a small number instead of 0

# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1

# Note: I am initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g. all zeros could mask an incorrect
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

4137


In [43]:
# creating a single mini batch for training
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
Yb

tensor([ 8, 14, 15, 22,  0, 19,  9, 14,  5,  1, 20,  3,  8, 14, 12,  0, 11,  0,
        26,  9, 25,  0,  1,  1,  7, 18,  9,  3,  5,  9,  0, 18])

In [44]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors

# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation

# BatchNorm layer - BatchNorm calculation splitted so as to check backprop manually
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias

# Non-linearity
h = torch.tanh(hpreact) # hidden layer

# Linear layer 2
logits = h @ W2 + b2 # output layer

# cross entropy loss (same as F.cross_entropy(logits, Yb)) - calculation splitted so as to check backprop manually
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean() # logprobs[range(n), Yb] extracts the log-probability value from the logprobs tensor that corresponds to the true next character (Yb[i])

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad() # Enables this Tensor to have their grad populated during backward()
loss.backward()
loss

tensor(3.3381, grad_fn=<NegBackward0>)

In [51]:
# Exercise 1: backprop through the whole thing manually,
# backpropagating through exactly all of the variables as they are defined in the forward pass above, one by one

### grads of loss calculation ###

# find derivative of logprobs(i.e dlogprobs) for equation 'loss = -logprobs[range(n), Yb].mean()'
# here we see loss is calculated by finding the negative of average/mean of probablities of corresponds true next char
# a simplified similar calculation loss = -(a + b + c)/3 . then applying sum-rule we get dloss/da = -1/3
dlogprobs = torch.zeros_like(logprobs) # init a zero tensor with shape of logprobs
dlogprobs[range(n), Yb] = -1.0/n
cmp('logprobs', dlogprobs, logprobs)

# find derivative of probs(i.e dprobs) for equation 'logprobs = probs.log()'
# The logprobs is calculated taking log of probs
# The LOCAL derivative dlogprobs/dprobs = 1/probs using log-rule https://www.wolframalpha.com/input?i=d%2Fdx+log%28x%29
# then dloss/dprobs = 1/probs * dlogprobs using chain rule
dprobs = (1.0 / probs) * dlogprobs
cmp('probs', dprobs, probs)

# find derivative of counts_sum_inv(i.e dcounts_sum_inv) for equation 'probs = counts * counts_sum_inv'
# The LOCAL derivative dprobs/dcounts_sum_inv = counts using https://www.wolframalpha.com/input?i=partial+derivative+of+x*y
# then dloss/dcounts_sum_inv = counts * dprobs using chain rule
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True) # for details https://youtu.be/q8SA3rM6ckI?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&t=1345
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

# find derivative of counts(i.e dcounts) for equation 'probs = counts * counts_sum_inv'
# The LOCAL derivative dprobs/dcounts = counts_sum_inv using https://www.wolframalpha.com/input?i=partial+derivative+of+x*y
# then dloss/dcounts = counts_sum_inv * dprobs using chain rule
dcounts = (counts_sum_inv * dprobs) # for details https://youtu.be/q8SA3rM6ckI?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&t=1611
#cmp('counts', dcounts, counts) # cant check it here as counts is used multiple times in two diff expression

# find derivative of counts_sum(i.e dcounts_sum) for equation 'counts_sum_inv = counts_sum**-1' or 'counts_sum_inv = 1.0 / counts_sum'
# The LOCAL derivative dcounts_sum_inv/dcounts_sum = -counts_sum**-2 or -1/counts_sum**2 using https://www.wolframalpha.com/input?i=d%2Fdx+x**-1
# then dloss/dcounts_sum = -counts_sum**-2 * dcounts_sum_inv using chain rule
dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv
cmp('counts_sum', dcounts_sum, counts_sum)

# find derivative of counts(i.e dcounts) for equation 'counts_sum = counts.sum(1, keepdims=True)'
# The LOCAL derivative ddcounts_sum/dcounts = 1 using https://www.wolframalpha.com/input?i=d%2Fdx+x%2By
# then dloss/dcounts = 1 * dcounts_sum using chain rule
dcounts += torch.ones_like(counts) * dcounts_sum # here we use += so as to add with the previously calculated dcounts
cmp('counts', dcounts, counts)

# find derivative of norm_logits(i.e dnorm_logits) for equation 'counts = norm_logits.exp()'
# The LOCAL derivative dcounts/dnorm_logits = norm_logits.exp() using exponent-rule
# then dloss/dnorm_logits = norm_logits.exp() * dcounts using chain rule
dnorm_logits = (norm_logits.exp()) * dcounts
cmp('norm_logits', dnorm_logits, norm_logits)

# find derivative of logits(i.e dlogits) for equation 'norm_logits = logits - logit_maxes'
# The LOCAL derivative dnorm_logits/dlogits = 1 using https://www.wolframalpha.com/input?i=d%2Fdx++x-y
# then dloss/dlogits = 1 * dnorm_logits using chain rule
dlogits = dnorm_logits.clone() # or (1.0) * dnorm_logits.clone()
# cmp('logits', dlogits, logits) # not the final derivative of logits

# find derivative of logit_maxes(i.e dlogit_maxes) for equation 'norm_logits = logits - logit_maxes'
# The LOCAL derivative dnorm_logits/dlogit_maxes = -1 using https://www.wolframalpha.com/input?i=d%2Fdy++x-y
# then dloss/dlogit_maxes = -1 * dnorm_logits using chain rule
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True) # or (-1.0) * (dnorm_logits).sum(1, keepdim=True)
cmp('logit_maxes', dlogit_maxes, logit_maxes)

# find derivative of logits(i.e dlogits) for equation 'logit_maxes = logits.max(1, keepdim=True).values'
# The LOCAL derivative dlogit_maxes/dlogits = 1
# then dloss/dlogits = 1 *dlogit_maxes using chain rule
# dlogits = torch.zeros_like(logits)
# dlogits[range(n), logits.max(1).indices] = 1 # setting the derivative of the logit with max value as 1. so in a singe row in dlogits , there will be one '1' and reste as '0'
# dlogits += dlogits * dlogit_maxes
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
cmp('logits', dlogits, logits) # for details https://youtu.be/q8SA3rM6ckI?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&t=2310

### grads of layer2 ###

# find derivative of h(i.e dh) for equation 'logits = h @ W2 + b2'
# The LOCAL derivative dlogits/dh = -
# then dloss/dh =  using chain rule



logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0


In [48]:
dprobs.shape
counts_sum_inv.shape

torch.Size([32, 1])